<a href="https://colab.research.google.com/github/DeepSadanand/RAGAS-Evaluation/blob/main/RAGAS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install weaviate-client openai ragas langchain

In [4]:
import requests
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import Weaviate

import weaviate
from weaviate.embedded import EmbeddedOptions

In [5]:
url="https://raw.githubusercontent.com/arungop/ambedkar_books/refs/heads/main/data/annihilation_of_caste.py.txt"

In [6]:
req = requests.get(url)

In [ ]:
req.text

In [8]:
with open("ambedkar.txt",'w') as f:
  f.write(req.text)

In [9]:
document = TextLoader("ambedkar.txt")
loader = document.load()

In [10]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 3000, chunk_overlap = 50)


In [11]:
text_chunks = text_splitter.split_documents(loader)

In [ ]:
len(text_chunks)

In [ ]:
text_chunks[1].page_content

In [14]:
from google.colab import userdata
openai_api_key = userdata.get('OPENAI_API_KEY')
embed_model = OpenAIEmbeddings(openai_api_key = openai_api_key)

<ipython-input-14-84d7437fa426>:3: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAIEmbeddings``.
  embed_model = OpenAIEmbeddings(openai_api_key = openai_api_key)


In [15]:

import weaviate
from weaviate import connect_to_custom
import weaviate
from weaviate.classes.init import Auth
import os

wcd_url = "XXXXX" #_____> from WCD console REST Endpoint
wcd_api_key = "XXXXXXX" # -----> from WCD colsole it is gRPC Endpoint

#OPENAI_API_KEY = "XXXXCCCZCZ" # you can type any random key here if you are not using the Open AI LLM
openai_api_key = openai_api_key # set into in OS environment

client = weaviate.connect_to_weaviate_cloud(
    cluster_url=wcd_url,  # Replace with your Weaviate Cloud URL
    auth_credentials=Auth.api_key(wcd_api_key),  # Replace with your Weaviate Cloud key
    headers={'X-OpenAI-Api-key': openai_api_key}  # Replace with your OpenAI API key
)

In [ ]:
!pip install langchain-weaviate

In [17]:
from langchain_weaviate.vectorstores import WeaviateVectorStore
from langchain.chains import RetrievalQAWithSourcesChain
from langchain_openai import OpenAI

# Two well-known applications for combining LLMs and vector stores are:

### Question answering
### Retrieval-augmented generation (RAG)

In [25]:
docsearch = WeaviateVectorStore.from_documents(
    documents=text_chunks,
    embedding = embed_model,
    client=client,
    index_name = "langchain"
)





In [27]:
retriever = docsearch.as_retriever(search_type="mmr")


In [28]:
from langchain_core.prompts import ChatPromptTemplate

template = """You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.
Question: {question}
Context: {context}
Answer:
"""
prompt = ChatPromptTemplate.from_template(template)

print(prompt)

input_variables=['context', 'question'] input_types={} partial_variables={} messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question}\nContext: {context}\nAnswer:\n"), additional_kwargs={})]


In [31]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
llm = ChatOpenAI(openai_api_key = openai_api_key, model="gpt-3.5-turbo", temperature=0)

rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)



In [ ]:
rag_chain.invoke("Ambedkar what said about religion?")

In [56]:

questions=["what did the ambedkar said about religion?",
           "What did the President say abou Intel's CEO?",
           "What did the President say about gun violence?"

           ]

In [67]:
groud_truth=[["Ambedkar said that what Hindus call religion is actually law or legalized class-ethics."],
              ["The president said that Pat Gelsinger is ready to increase Intel's investment to $100 billion."],
              ["The president asked Congress to pass proven measures to reduce gun violence."]]

In [68]:
answer=[]
context=[]

for query in questions:
  answer.append(rag_chain.invoke(query))
  context.append([docs.page_content  for docs in retriever.get_relevant_documents(query)])

In [ ]:
answer

In [70]:
data = {
    "question": questions,
    "answer": answer,
    "contexts": context,
    "ground_truths": groud_truth,
    "reference":"NaN"

}


In [71]:
from datasets import Dataset
dataset=Dataset.from_dict(data)

In [75]:
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_recall,
    context_precision
)

In [76]:
import os
os.environ["OPENAI_API_KEY"] = openai_api_key

In [ ]:
result= evaluate(
    dataset=dataset,
    metrics=[
        context_precision,
        context_recall,
        faithfulness,
        answer_relevancy
    ]
)

In [66]:
result

{'context_precision': 0.0000, 'context_recall': 0.0000, 'faithfulness': 0.3333, 'answer_relevancy': 0.3024}

In [78]:
!git add .

fatal: not a git repository (or any of the parent directories): .git
